# 🧪 Practice Lab: Probability for AI Engineers

**Netsetos GenAI Course** | Module 1 · Lesson 1.8 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+, numpy, scipy (pre-installed in Colab)

### 🎯 You will:
1. Build PMFs for discrete distributions, verify sum = 1.0
2. Compare continuous distributions using percentiles
3. Implement temperature-controlled token sampling
4. Model API latency and build SLO checker
5. Build spectrogram from scratch
6. Full probability dashboard combining all distributions

---

## Exercise 1 (Easy): Discrete PMF Builder

**📝 Task:** Bernoulli, Binomial, Poisson, Categorical PMFs. Verify sum=1.0. 10K samples vs theory.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from scipy import stats
np.random.seed(42)
# TODO: build 4 PMFs, verify, compare

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

# Bernoulli
print(f'Bernoulli(p=0.3): P(0)=0.700, P(1)=0.300, sum={0.7+0.3:.3f}')
print(f'  Empirical (10K): P(1)={np.random.binomial(1,0.3,10000).mean():.3f}')

# Binomial
k = np.arange(0, 11)
pmf = stats.binom.pmf(k, 10, 0.3)
print(f'\nBinomial(n=10,p=0.3): sum={pmf.sum():.6f}')
samp = np.random.binomial(10, 0.3, 10000)
for v in [0,2,3,4,6]: print(f'  k={v}: theory={pmf[v]:.3f} emp={np.mean(samp==v):.3f}')

# Poisson
pmf_p = stats.poisson.pmf(np.arange(15), 4)
print(f'\nPoisson(λ=4): sum={pmf_p.sum():.6f}')

# Categorical
logits = np.array([2.1, 3.5, 0.8, 1.2, -0.3, 0.5])
pmf_c = np.exp(logits)/np.exp(logits).sum()
print(f'\nCategorical: sum={pmf_c.sum():.6f}')
for t, p in zip(['the','cat','sat','on','mat','dog'], pmf_c):
    print(f'  {t:6s} {p:.4f} {"█"*int(p*30)}')
print('\n✅ All PMFs verified')

</details>

---

## Exercise 2 (Easy): Continuous PDF + Percentiles

**📝 Task:** Gaussian, Log-Normal, Exponential, Beta. mean vs median gap. Why percentiles matter.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)
# TODO: 4 distributions, percentiles, gap analysis

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

dists = {
    'Gaussian': np.random.randn(10000),
    'Log-Normal': np.random.lognormal(5.5, 0.6, 10000),
    'Exponential': np.random.exponential(200, 10000),
    'Beta': np.random.beta(2, 5, 10000),
}

print(f'{"Dist":<14}{"mean":>8}{"median":>8}{"Gap":>8}{"p95":>8}{"p99":>8}')
print('-'*58)
for n, s in dists.items():
    m, md = s.mean(), np.median(s)
    print(f'  {n:<12}{m:>7.1f}{md:>7.1f}{m-md:>7.1f}{np.percentile(s,95):>7.1f}{np.percentile(s,99):>7.1f}')
print('\nLog-Normal/Exponential: large gap → use percentiles, not mean!')

</details>

---

## Exercise 3 (Medium): Token Prediction + Temperature

**📝 Task:** 20-word vocab, softmax PMF, temperature 0.1/0.5/1.0/2.0, entropy, generate sequences.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)
vocab = [f'w{i}' for i in range(20)]
logits = np.random.randn(20) * 2
# TODO: softmax, temperature, entropy, generate

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)
vocab = [f'w{i}' for i in range(20)]
logits = np.random.randn(20) * 2

def softmax_t(l, T):
    s = l/max(T,1e-8)
    e = np.exp(s-s.max())
    return e/e.sum()

def entropy(p):
    p = p[p>0]
    return -np.sum(p*np.log2(p))

for T in [0.1, 0.5, 1.0, 2.0]:
    pmf = softmax_t(logits, T)
    h = entropy(pmf)
    seq = ' '.join(np.random.choice(vocab, size=10, p=pmf))
    print(f'T={T}: entropy={h:.2f}bits top1={pmf.max():.3f} → {seq}')
print('\nT↑ → entropy↑ → more random')

</details>

---

## Exercise 4 (Medium): Log-Normal Latency + SLO

**📝 Task:** 10K latencies, p50/p95/p99, SLO checker (p99<1000ms), add 2% spikes, re-check.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
np.random.seed(42)
latencies = np.random.lognormal(5.5, 0.6, 10000)
# TODO: stats, SLO, spikes

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)
lat = np.random.lognormal(5.5, 0.6, 10000)

def check(data, label):
    p50,p95,p99 = [np.percentile(data,p) for p in [50,95,99]]
    slo = p99 < 1000
    print(f'\n{label}:')
    print(f'  mean={data.mean():.0f}ms median={np.median(data):.0f}ms')
    print(f'  p50={p50:.0f} p95={p95:.0f} p99={p99:.0f}')
    print(f'  SLO(p99<1000): {"PASS✅" if slo else "FAIL❌"}')
    print(f'  mean alert(<500): PASS' if data.mean()<500 else '  mean alert: PASS')

check(lat, 'Normal')
spk = lat.copy()
spk[np.random.choice(len(spk), int(len(spk)*0.02), replace=False)] *= 5
check(spk, 'With 2% spikes')
print('\n→ mean barely moved but p99 blew up. Use percentiles!')

</details>

---

## Exercise 5 (Hard): Spectrogram from Scratch

**📝 Task:** 1s audio (3 freqs + noise), STFT, magnitude spectrogram, mel binning. Whisper's input format.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
sr = 16000
t = np.arange(0, 1.0, 1/sr)
signal = np.sin(2*np.pi*440*t) + 0.5*np.sin(2*np.pi*880*t) + 0.3*np.sin(2*np.pi*1320*t)
signal += np.random.randn(len(t))*0.1
# TODO: STFT, spectrogram, mel binning

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
sr = 16000
t = np.arange(0, 1.0, 1/sr)
signal = np.sin(2*np.pi*440*t) + 0.5*np.sin(2*np.pi*880*t) + 0.3*np.sin(2*np.pi*1320*t)
signal += np.random.randn(len(t))*0.1

win = int(0.025*sr); hop = int(0.010*sr)
nf = (len(signal)-win)//hop + 1
spec = []
for i in range(nf):
    frame = signal[i*hop:i*hop+win] * np.hanning(win)
    spec.append(np.abs(np.fft.rfft(frame)))
spec = np.array(spec)
print(f'Spectrogram: {spec.shape} = {nf} frames × {spec.shape[1]} freq bins')

freqs = np.fft.rfftfreq(win, 1/sr)
for fi in [0, nf//2]:
    top = freqs[spec[fi].argsort()[-3:][::-1]].astype(int)
    print(f'  Frame {fi}: dominant = {top} Hz')

# mel binning
n_mels = 40
bpm = spec.shape[1]//n_mels
mel = np.zeros((nf, n_mels))
for m in range(n_mels):
    mel[:,m] = spec[:, m*bpm:min((m+1)*bpm, spec.shape[1])].mean(axis=1)
log_mel = np.log(mel+1e-10)
print(f'\nmel spectrogram: {log_mel.shape} = Whisper input format!')

</details>

---

## Exercise 6 (Hard): Full Probability Pipeline

**📝 Task:** All 5 distributions → unified production dashboard with SLOs and alerts.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from scipy import stats
np.random.seed(42)
# TODO: all distributions, unified dashboard

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print('═'*50)
print('  PROBABILITY DASHBOARD')
print('═'*50)

# 1. Categorical
logits = np.random.randn(1000)*2
pmf = np.exp(logits)/np.exp(logits).sum()
H = -np.sum(pmf[pmf>0]*np.log2(pmf[pmf>0]))
print(f'\n1. Tokens: entropy={H:.1f}bits top5={np.sort(pmf)[-5:].sum()*100:.1f}%')

# 2. Gaussian
emb = np.random.randn(768)
print(f'2. Embedding: μ={emb.mean():.3f} σ={emb.std():.3f} {"✅" if abs(emb.mean())<0.1 else "❌"}')

# 3. Log-Normal
lat = np.random.lognormal(5.5, 0.6, 1000)
p99 = np.percentile(lat, 99)
print(f'3. Latency: p50={np.percentile(lat,50):.0f} p95={np.percentile(lat,95):.0f} p99={p99:.0f}ms SLO:{"✅" if p99<1000 else "❌"}')

# 4. Poisson
err = np.random.poisson(3, 1000)
print(f'4. Errors: mean={err.mean():.1f}/hr P(>5)={np.mean(err>5):.3f} {"✅" if np.mean(err>5)<0.1 else "⚠️"}')

# 5. Beta
pa = stats.beta(15+1, 185+1)
pb = stats.beta(22+1, 178+1)
pbw = np.mean(pb.rvs(10000)>pa.rvs(10000))
print(f'5. A/B Test: P(B>A)={pbw:.3f} {"HIGH" if pbw>0.9 else "MOD" if pbw>0.8 else "LOW"}')

print(f'\n{"═"*50}')
print(f'  All systems nominal ✅')
print(f'{"═"*50}')

</details>

---

## 🎉 Module 1 (classical-ML track) Complete!

You've mastered all probability distributions for GenAI:
- **Categorical** — token prediction
- **Gaussian** — embeddings
- **Log-Normal** — latency (use percentiles!)
- **Poisson** — error rates
- **Beta** — A/B testing
- **Spectrogram** — audio/speech

These foundations power **Modules 4-13.** Ready for **Module 3!**